# Lab 2.6: Information Provenance & Multi-Source Synthesis

**What you'll build:** A provenance-tracked research synthesis system that maps every claim to its source — and learn why coherent summaries can hide dangerous conflicts.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way — naive synthesis silently picks favorites among conflicting sources | 5 min |
| 2 | The Right Way — claim-source mappings with conflict detection and gap reporting | 5 min |
| 3 | Your Turn — build provenance tracking for a multi-source research query | 10 min |
| 4 | Stress Test — verify conflict detection consistency across runs | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You're building a research synthesis system that pulls data from multiple sources about companies. The challenge: sources often conflict (different revenue numbers, different employee counts) and some topics aren't covered by any source.

A naive synthesis picks one value and presents it as fact. A provenance-tracked synthesis maps every claim to its source, surfaces conflicts, and reports gaps.

In [ ]:
# Research sources with conflicts and gaps
RESEARCH_SOURCES = {
    "annual_report": {
        "source_name": "Acme Corp 2024 Annual Report",
        "source_date": "2024-03-15",
        "source_type": "official_filing",
        "data": {
            "revenue": "$4.2 billion",
            "employees": 15000,
            "headquarters": "San Francisco, CA",
            "founded": 2010,
            "ceo": "Jane Smith"
        }
    },
    "news_article": {
        "source_name": "TechCrunch article: 'Acme's Big Year'",
        "source_date": "2024-12-01",
        "source_type": "news",
        "data": {
            "revenue": "$4.5 billion (projected)",
            "employees": 16200,
            "new_office": "Austin, TX",
            "market_share": "23%"
        }
    },
    "analyst_report": {
        "source_name": "Goldman Sachs Equity Research",
        "source_date": "2024-09-20",
        "source_type": "analyst",
        "data": {
            "revenue": "$4.3 billion (estimated)",
            "employees": 15500,
            "market_share": "21%",
            "growth_rate": "18% YoY"
        }
    }
}

# What the user wants to know
RESEARCH_QUERY = "Provide a comprehensive profile of Acme Corp covering: revenue, employee count, market share, headquarters, ESG rating, and growth rate."

# Ground truth: what correct provenance tracking should produce
EXPECTED_CONFLICTS = [
    {"topic": "revenue", "values": ["$4.2B", "$4.5B", "$4.3B"]},
    {"topic": "employees", "values": ["15000", "16200", "15500"]},
    {"topic": "market_share", "values": ["23%", "21%"]}
]

EXPECTED_GAPS = ["ESG rating"]  # Not in any source

print(f"Research query: {RESEARCH_QUERY}")
print(f"\nSources available: {len(RESEARCH_SOURCES)}")
for key, source in RESEARCH_SOURCES.items():
    print(f"  {source['source_name']} ({source['source_date']})")
print(f"\nExpected conflicts: {len(EXPECTED_CONFLICTS)}")
print(f"Expected gaps: {EXPECTED_GAPS}")

---

## Phase 1: The Wrong Approach

Naive synthesis: produce a coherent summary by silently resolving conflicts and ignoring gaps.

In [ ]:
NAIVE_SYNTHESIS_PROMPT = f"""You are a research analyst. Synthesize information from these sources
into a clear, coherent company profile.

Sources:
{json.dumps(RESEARCH_SOURCES, indent=2)}

Query: {RESEARCH_QUERY}

Write a professional company profile paragraph."""

response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    messages=[{"role": "user", "content": NAIVE_SYNTHESIS_PROMPT}]
)

naive_synthesis = response.content[0].text
print("=== NAIVE SYNTHESIS ===")
print(naive_synthesis)
print()
print("PROBLEMS WITH THIS:")
print("  1. Which revenue number did it pick? Was the choice justified?")
print("  2. Does the reader know sources disagree on employee count?")
print("  3. Was the ESG rating gap reported or silently omitted?")
print("  4. Can the reader verify any claim against a specific source?")

### Why naive synthesis is dangerous

- **Silently resolves conflicts.** The reader doesn't know sources disagree — they see one number as fact.
- **No claim-source mapping.** No way to verify which source a specific number came from.
- **Omits gaps without reporting.** The reader assumes ESG rating was checked and not found relevant, rather than "not available in any source."
- **Temporal conflicts hidden.** $4.2B (March) vs $4.5B (December) may reflect growth, not disagreement — but the reader can't evaluate without dates.

---

## Phase 2: The Right Approach

Provenance-tracked synthesis: map every claim to its source, surface conflicts, report gaps.

In [ ]:
PROVENANCE_SYNTHESIS_PROMPT = f"""You are a research analyst. Synthesize information from these sources
with full provenance tracking.

Sources:
{json.dumps(RESEARCH_SOURCES, indent=2)}

Query: {RESEARCH_QUERY}

PROVENANCE RULES:
1. Map every claim to its source(s) with date.
2. When sources AGREE on a value, cite all confirming sources.
3. When sources CONFLICT, report ALL values with their sources. Do NOT silently pick one.
4. For temporal data, note the date of each source and whether recency explains the difference.
5. EXPLICITLY report any requested topic that was NOT found in any source (coverage gaps).
6. Do NOT fabricate data to fill gaps.

Respond as JSON: {{
  "claims": [
    {{
      "topic": "...",
      "value": "..." or null if conflicting,
      "sources": [{{"name": "...", "date": "...", "value": "..."}}],
      "status": "confirmed" or "conflicting" or "single_source" or "not_found",
      "conflict_note": "explanation if conflicting" or null
    }}
  ],
  "coverage_gaps": ["topics not found in any source"],
  "overall_confidence": "high/medium/low"
}}"""

response = client.messages.create(
    model=MODEL,
    max_tokens=1200,
    messages=[{"role": "user", "content": PROVENANCE_SYNTHESIS_PROMPT}]
)

provenance_result = response.content[0].text
print("=== PROVENANCE-TRACKED SYNTHESIS ===")
print(provenance_result)

In [ ]:
# Check: did the provenance system catch the conflicts and gaps?
raw = provenance_result.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    result = json.loads(raw)
    
    # Check for conflicts
    conflicting_topics = [c["topic"] for c in result.get("claims", []) if c.get("status") == "conflicting"]
    print("Conflicts detected:")
    for t in conflicting_topics:
        print(f"  - {t}")
    
    # Check for gaps
    gaps = result.get("coverage_gaps", [])
    print(f"\nCoverage gaps reported: {gaps}")
    
    # Verify
    found_revenue_conflict = any("revenue" in t.lower() for t in conflicting_topics)
    found_esg_gap = any("esg" in g.lower() for g in gaps)
    
    print(f"\n=== PROVENANCE QUALITY CHECK ===")
    print(f"  Revenue conflict detected: {'YES' if found_revenue_conflict else 'NO'}")
    print(f"  ESG gap reported: {'YES' if found_esg_gap else 'NO'}")
    
except json.JSONDecodeError:
    print("Failed to parse provenance JSON. Check prompt output format.")

---

## Phase 3: Your Turn

Build a provenance-tracked synthesis for a different research query with its own conflicts and gaps.

In [ ]:
CHALLENGE_SOURCES = {
    "government_filing": {
        "source_name": "SEC 10-K Filing",
        "source_date": "2024-02-28",
        "source_type": "regulatory_filing",
        "data": {
            "annual_revenue": "$2.1 billion",
            "net_income": "$310 million",
            "total_assets": "$5.8 billion",
            "employees": 8500,
            "debt_ratio": 0.35
        }
    },
    "company_press_release": {
        "source_name": "NovaTech Q3 2024 Press Release",
        "source_date": "2024-10-15",
        "source_type": "company_statement",
        "data": {
            "annual_revenue": "$2.4 billion (projected)",
            "employees": 9200,
            "new_product": "NovaAI Platform",
            "customer_count": 12000
        }
    },
    "industry_database": {
        "source_name": "Bloomberg Terminal Data",
        "source_date": "2024-11-30",
        "source_type": "data_provider",
        "data": {
            "annual_revenue": "$2.3 billion (TTM)",
            "market_cap": "$18.5 billion",
            "pe_ratio": 42.3,
            "employees": 9100,
            "sector_rank": 5
        }
    }
}

CHALLENGE_QUERY = "Provide a comprehensive profile of NovaTech covering: annual revenue, employee count, net income, market cap, customer count, ESG score, and debt ratio."

# Expected results
EXPECTED = {
    "conflicts": ["annual_revenue", "employees"],
    "gaps": ["ESG score"],
    "single_source": ["net_income", "customer_count", "market_cap", "debt_ratio"]
}

print(f"Challenge query: {CHALLENGE_QUERY}")
print(f"\nExpected conflicts: {EXPECTED['conflicts']}")
print(f"Expected gaps: {EXPECTED['gaps']}")
print(f"Expected single-source: {EXPECTED['single_source']}")

In [ ]:
# =============================================================
# TODO: Write a provenance synthesis prompt
# =============================================================
#
# Requirements:
# - Every claim must map to its source(s)
# - Conflicts must be surfaced with all values and sources
# - Temporal differences must be noted
# - Coverage gaps must be explicitly reported
# - Must NOT fabricate data for missing topics
#
# Think about:
# - How do you instruct the model to detect conflicts?
# - How do you prevent the model from smoothing over disagreements?
# - How do you ensure gaps are reported rather than omitted?

YOUR_PROVENANCE_PROMPT = f"""You are a research analyst.

# TODO: Write your provenance tracking instructions here.
# Map claims to sources, detect conflicts, report gaps.

Sources:
{json.dumps(CHALLENGE_SOURCES, indent=2)}

Query: {CHALLENGE_QUERY}

Respond as JSON: {{{{
  "claims": [
    {{
      "topic": "...",
      "value": "...",
      "sources": [{{"name": "...", "date": "...", "value": "..."}}],
      "status": "confirmed" or "conflicting" or "single_source" or "not_found"
    }}
  ],
  "coverage_gaps": ["topics not found"],
  "overall_confidence": "high/medium/low"
}}}}"""

response = client.messages.create(
    model=MODEL,
    max_tokens=1200,
    messages=[{"role": "user", "content": YOUR_PROVENANCE_PROMPT}]
)

your_result_text = response.content[0].text
print("=== YOUR PROVENANCE RESULT ===")
print(your_result_text)

In [ ]:
# =============================================================
# Checker: validates your provenance tracking
# =============================================================

raw = your_result_text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    result = json.loads(raw)
    claims = result.get("claims", [])
    gaps = result.get("coverage_gaps", [])
    
    # Check conflicts detected
    conflicting = [c["topic"].lower() for c in claims if c.get("status") == "conflicting"]
    found_revenue_conflict = any("revenue" in t for t in conflicting)
    found_employee_conflict = any("employee" in t for t in conflicting)
    
    # Check gaps
    found_esg_gap = any("esg" in g.lower() for g in gaps)
    
    # Check that claims have sources
    claims_with_sources = all(len(c.get("sources", [])) > 0 for c in claims if c.get("status") != "not_found")
    
    checks = {
        "revenue_conflict_detected": found_revenue_conflict,
        "employee_conflict_detected": found_employee_conflict,
        "esg_gap_reported": found_esg_gap,
        "all_claims_have_sources": claims_with_sources
    }
    
    print("=== YOUR SCORECARD ===")
    for check, passed in checks.items():
        status = "PASS" if passed else "FAIL"
        print(f"  {check}: {status}")
    
    if all(checks.values()):
        print("\n  PASSED — provenance tracking correctly identifies conflicts and gaps!")
    else:
        failed = [c for c, p in checks.items() if not p]
        print(f"\n  NEEDS WORK: {failed}")
        
except json.JSONDecodeError:
    print("Failed to parse JSON. Check your prompt's output format.")

---

## Phase 4: Stress Test

Verify conflict detection is consistent across runs.

In [ ]:
print("Running provenance tracking 5 times...\n")

consistency_results = []
for run in range(5):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1200,
        messages=[{"role": "user", "content": YOUR_PROVENANCE_PROMPT}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        result = json.loads(raw)
        conflicting = [c["topic"].lower() for c in result.get("claims", []) if c.get("status") == "conflicting"]
        gaps = [g.lower() for g in result.get("coverage_gaps", [])]
        score = (
            any("revenue" in t for t in conflicting) +
            any("employee" in t for t in conflicting) +
            any("esg" in g for g in gaps)
        )
        consistency_results.append(score)
        print(f"  Run {run+1}: {score}/3 checks passed (conflicts: {conflicting}, gaps: {gaps})")
    except json.JSONDecodeError:
        consistency_results.append(0)
        print(f"  Run {run+1}: Failed to parse")

print(f"\n=== CONSISTENCY ===")
print(f"Scores: {consistency_results}")
if all(r == 3 for r in consistency_results):
    print("Perfect consistency — provenance tracking is reliable.")
else:
    print("Inconsistent — strengthen your conflict detection and gap reporting instructions.")

### Key Takeaways

1. **Every claim needs a source.** Claim-source mappings make outputs verifiable and trustworthy.
2. **Conflicts must be surfaced, not smoothed.** Coherent summaries hide disagreements that the reader needs to evaluate.
3. **Temporal context matters.** Note source dates — a difference between March and December data may be growth, not error.
4. **Coverage gaps must be explicitly reported.** Omitting unfound topics creates a false sense of completeness.
5. **The reader needs to evaluate, not just consume.** Provenance tracking gives readers the evidence to form their own judgment.